# Harmony fine-tuning (Vision+Text) — `unsloth/medgemma-4b-it-bnb-4bit`

This notebook fine-tunes **MedGemma 4B IT (4-bit)** for **image+text → text** using a Harmony-style JSONL where each example contains:

- `image_path`: local path to the image (e.g. chest X-ray)
- `messages`: chat turns (user/assistant)

We use **LoRA (PEFT)** on top of the 4-bit base model.


In [1]:
# -----------------------------
# Hugging Face auth (required if model is gated)
# -----------------------------
from huggingface_hub import login

token_path = "/data/liangz2/openi/hf_token.txt"
with open(token_path, "r") as f:
    hf_token = f.readline().strip()

login(token=hf_token)
print("✅ Hugging Face login successful.")


✅ Hugging Face login successful.


In [2]:
import os, json, csv
from dataclasses import dataclass
from typing import Any, Dict, List

import torch
from torch.utils.data import Dataset

from PIL import Image

from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    TrainingArguments,
    Trainer,
)

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training


In [3]:
# -----------------------------
# Config
# -----------------------------
MODEL_ID = "unsloth/medgemma-4b-it-bnb-4bit"

JSONL_PATH = "/data/liangz2/openi/harmony_set/openi_cxr_harmony_rl.jsonl"  # update if needed
OUTPUT_DIR = "/data/liangz2/openi/medgemma_4b"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MAX_LEN = 1536          # adjust for your GPU
SEED = 42
TRAIN_FRAC = 0.98

# LoRA
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05

# Generation (for quick sanity checks)
GEN_MAX_NEW_TOKENS = 128


In [4]:
# -----------------------------
# Load processor + model
# -----------------------------
processor = AutoProcessor.from_pretrained(MODEL_ID)

# Important for batching
if getattr(processor, "tokenizer", None) is not None and processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    device_map="auto",
)

# Prepare for k-bit training + attach LoRA
model = prepare_model_for_kbit_training(model)

# MedGemma/Gemma-style modules are usually compatible with these targets.
# If you hit "module not found", see the note cell below for how to inspect module names.
target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=target_modules,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.23G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

trainable params: 65,576,960 || all params: 4,365,656,432 || trainable%: 1.5021


## Note: If LoRA target modules error

If you see an error like `Target modules ... not found`, inspect module names:

```python
for name, _ in model.named_modules():
    if any(k in name for k in ["q_proj","k_proj","v_proj","o_proj"]):
        print(name)
```

Then update `target_modules` accordingly.


In [7]:
# -----------------------------
# Helpers: parse Harmony JSONL
# -----------------------------
def _extract_text_from_content(content) -> str:
    if content is None:
        return ""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        out = []
        for part in content:
            if isinstance(part, str):
                out.append(part)
            elif isinstance(part, dict):
                if "text" in part and isinstance(part["text"], str):
                    out.append(part["text"])
                elif "content" in part and isinstance(part["content"], str):
                    out.append(part["content"])
                else:
                    pass
        return "\n".join([x for x in out if x.strip()])
    if isinstance(content, dict):
        if "text" in content and isinstance(content["text"], str):
            return content["text"]
        if "content" in content and isinstance(content["content"], str):
            return content["content"]
        return json.dumps(content, ensure_ascii=False)
    return str(content)

def _normalize_messages(msgs):
    norm = []
    for m in msgs or []:
        if not isinstance(m, dict):
            continue
        role = (m.get("role") or "").strip().lower()
        if role not in {"system", "user", "assistant"}:
            role = "user"
        content = _extract_text_from_content(m.get("content"))
        norm.append({"role": role, "content": content})
    return norm


import json
from typing import Any, Dict, List

def read_harmony_jsonl_with_images(jsonl_path: str) -> List[Dict[str, Any]]:
    """
    Reads Harmony JSONL with:
      - image_path (absolute path)
      - messages (system/user turns)
      - ground_truth.answer ("TRUE"/"FALSE")  <-- used to create assistant supervision
    Returns rows with a synthesized assistant turn if missing.
    """
    rows: List[Dict[str, Any]] = []
    skipped_no_image = 0
    skipped_no_msgs = 0
    skipped_no_gt = 0

    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            ex = json.loads(line)
            img_path = ex.get("image_path", None)
            msgs = ex.get("messages", None)

            if not img_path:
                skipped_no_image += 1
                continue
            if not isinstance(msgs, list) or len(msgs) == 0:
                skipped_no_msgs += 1
                continue

            msgs = _normalize_messages(msgs)

            # If the JSONL doesn't include an assistant message, synthesize it from ground_truth.answer
            if not any(m["role"] == "assistant" for m in msgs):
                gt = ex.get("ground_truth", {})
                ans = gt.get("answer") if isinstance(gt, dict) else None
                if not ans:
                    skipped_no_gt += 1
                    continue
                msgs.append({"role": "assistant", "content": str(ans).strip()})

            rows.append({"image_path": img_path, "messages": msgs})

    print(
        f"Loaded rows={len(rows)} | skipped(no_image)={skipped_no_image} "
        f"| skipped(no_messages)={skipped_no_msgs} | skipped(no_ground_truth)={skipped_no_gt}"
    )
    return rows


In [8]:
# -----------------------------
# Build train/eval splits
# -----------------------------
import random
random.seed(SEED)

all_rows = read_harmony_jsonl_with_images(JSONL_PATH)
print("Total rows with image_path+messages:", len(all_rows))
if len(all_rows) == 0:
    raise ValueError("No usable rows found. Verify JSONL_PATH and that records contain 'image_path' + 'messages'.")

random.shuffle(all_rows)
n_train = int(len(all_rows) * TRAIN_FRAC)
train_rows = all_rows[:n_train]
eval_rows = all_rows[n_train:]

print("Train:", len(train_rows), "Eval:", len(eval_rows))


Loaded rows=7902 | skipped(no_image)=0 | skipped(no_messages)=0 | skipped(no_ground_truth)=0
Total rows with image_path+messages: 7902
Train: 7743 Eval: 159


In [12]:
# -----------------------------
# Dataset: creates (input_ids, pixel_values, labels)
# -----------------------------

from PIL import Image
import torch
from torch.utils.data import Dataset
from typing import Any, Dict, List

def get_image_token(processor) -> str:
    tok = getattr(processor, "tokenizer", None)
    if tok is not None and hasattr(tok, "image_token") and isinstance(tok.image_token, str):
        return tok.image_token
    return "<image>"  # common fallback

def find_image_token(processor) -> str:
    """
    Gemma3/MedGemma processors require *the exact* special image placeholder token
    to appear in the prompt text. This function tries to discover it from the tokenizer.
    """
    tok = getattr(processor, "tokenizer", None)
    if tok is None:
        raise ValueError("Processor has no tokenizer; cannot discover image token.")

    v = getattr(tok, "image_token", None)
    if isinstance(v, str) and v:
        return v

    specials = list(getattr(tok, "all_special_tokens", []) or [])
    candidates = [t for t in specials if ("image" in t.lower()) or ("img" in t.lower())]
    if candidates:
        candidates.sort(key=len)
        return candidates[0]

    raise ValueError(
        "Could not find an image token in tokenizer special tokens. "
        "Print processor.tokenizer.all_special_tokens to inspect."
    )


# -----------------------------
# Dataset: creates (input_ids, pixel_values, labels)
# True/False verification: supervise only the assistant label tokens ("TRUE"/"FALSE")
# -----------------------------
class HarmonyVisionTextDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]], processor, max_len: int = 1536):
        self.rows = rows
        self.processor = processor
        self.max_len = max_len
        self.image_token = find_image_token(processor)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        ex = self.rows[idx]
        image_path = ex["image_path"]
        messages = ex["messages"]

        image = Image.open(image_path).convert("RGB")

        last_user_idx = max(i for i, m in enumerate(messages) if m["role"] == "user")
        prompt_only = messages[: last_user_idx + 1]
        full_conv = messages

        def build_text(msgs, add_generation_prompt: bool) -> str:
            mm = []
            for i, m in enumerate(msgs):
                role = m["role"]
                content = m["content"]
                if role == "user" and i == 0:
                    content = f"{self.image_token}
{content}"
                mm.append({"role": role, "content": content})
            return self.processor.apply_chat_template(
                mm,
                add_generation_prompt=add_generation_prompt,
                tokenize=False,
            )

        prompt_text = build_text(prompt_only, add_generation_prompt=True)
        full_text = build_text(full_conv, add_generation_prompt=False)

        enc_prompt = self.processor(
            text=prompt_text,
            images=image,
            return_tensors="pt",
            truncation=True,
            max_length=self.max_len,
        )
        enc_full = self.processor(
            text=full_text,
            images=image,
            return_tensors="pt",
            truncation=True,
            max_length=self.max_len,
        )

        input_ids = enc_full["input_ids"][0]
        attention_mask = enc_full.get("attention_mask", None)
        if attention_mask is not None:
            attention_mask = attention_mask[0]

        pixel_values = enc_full["pixel_values"][0]

        prompt_len = int(enc_prompt["input_ids"].shape[1])
        labels = input_ids.clone()
        labels[:prompt_len] = -100

        out = {"input_ids": input_ids, "labels": labels, "pixel_values": pixel_values}
        if attention_mask is not None:
            out["attention_mask"] = attention_mask
        return out


In [13]:
# -----------------------------
# Data collator (pads input_ids/labels, stacks pixel_values)
# -----------------------------
@dataclass
class VLDataCollator:
    processor: Any
    label_pad_token_id: int = -100

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        pixel_values = None
        if "pixel_values" in features[0]:
            pixel_values = torch.stack([f["pixel_values"] for f in features], dim=0)

        tok = self.processor.tokenizer if hasattr(self.processor, "tokenizer") else self.processor

        input_ids = [f["input_ids"] for f in features]
        attention_mask = [f.get("attention_mask", torch.ones_like(f["input_ids"])) for f in features]
        labels = [f["labels"] for f in features]

        batch = tok.pad(
            {"input_ids": input_ids, "attention_mask": attention_mask},
            padding=True,
            return_tensors="pt",
        )

        max_len = batch["input_ids"].shape[1]
        padded_labels = torch.full((len(labels), max_len), self.label_pad_token_id, dtype=torch.long)
        for i, lab in enumerate(labels):
            padded_labels[i, : lab.shape[0]] = lab

        batch["labels"] = padded_labels
        if pixel_values is not None:
            batch["pixel_values"] = pixel_values
        return batch


In [15]:
train_ds = HarmonyVisionTextDataset(train_rows, processor, max_len=MAX_LEN)
eval_ds = HarmonyVisionTextDataset(eval_rows, processor, max_len=MAX_LEN) if len(eval_rows) else None

data_collator = VLDataCollator(processor=processor)

# print("Tokenizer special tokens:", processor.tokenizer.all_special_tokens)
print("Image token used:", train_ds.image_token)
sample = train_ds[0]
# print({k: (v.shape, v.dtype) for k, v in sample.items()})


In [16]:
print("Image token used:", train_ds.image_token)
print("Tokenizer special tokens:", processor.tokenizer.all_special_tokens)
print("Image token used:", train_ds.image_token)
sample = train_ds[0]
print({k: (v.shape, v.dtype) for k, v in sample.items()})

Image token used: <image_soft_token>


ValueError: Prompt contained 0 image tokens but received 1 images.

In [ ]:
# -----------------------------
# TrainingArguments
# -----------------------------
do_eval = eval_ds is not None and len(eval_ds) > 0

training_args = TrainingArguments(
    output_dir=os.path.join(OUTPUT_DIR, "checkpoints"),
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.0,
    logging_steps=10,
    save_steps=1000,
    eval_steps=1000 if do_eval else None,
    eval_strategy="steps" if do_eval else "no",
    save_strategy="steps",
    save_total_limit=2,
    bf16=torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8,
    fp16=torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] < 8,
    gradient_checkpointing=True,
    report_to=[],
    dataloader_num_workers=4,
    optim="paged_adamw_8bit",
    remove_unused_columns=False,
    seed=SEED,
)
print(training_args)


In [ ]:
# -----------------------------
# Trainer
# -----------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
)

train_result = trainer.train()
print(train_result)


In [ ]:
# -----------------------------
# Save LoRA adapter + processor
# -----------------------------
def save_lora_adapter(model, processor, output_dir, adapter_name="lora"):
    save_path = os.path.join(output_dir, adapter_name)
    os.makedirs(save_path, exist_ok=True)

    model.save_pretrained(save_path)
    processor.save_pretrained(save_path)
    print(f"✅ LoRA adapter saved to: {save_path}")

save_lora_adapter(trainer.model, processor, OUTPUT_DIR)


In [ ]:
# -----------------------------
# Save metrics to CSV
# -----------------------------
METRICS_CSV = os.path.join(OUTPUT_DIR, "training_metrics.csv")
log_history = trainer.state.log_history

if not log_history:
    print("⚠️ No metrics found in trainer.state.log_history")
else:
    fieldnames = sorted({k for row in log_history for k in row.keys()})
    with open(METRICS_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in log_history:
            writer.writerow(row)
    print("✅ Saved metrics:", METRICS_CSV)


In [ ]:
# -----------------------------
# Quick inference sanity check (one sample)
# -----------------------------
ex = eval_rows[0] if len(eval_rows) else train_rows[0]
image = Image.open(ex["image_path"]).convert("RGB")
user_text = next(m["content"] for m in ex["messages"] if m["role"] == "user")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": user_text},
        ]
    }
]

prompt_text = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
enc = processor(text=prompt_text, images=image, return_tensors="pt").to(model.device)

with torch.no_grad():
    out = model.generate(**enc, max_new_tokens=3, do_sample=False)

print(processor.decode(out[0][enc["input_ids"].shape[-1]:], skip_special_tokens=True))
